The implementations for stability selection are quite dated, so I am adapting the simple function shared in [Thomas Huijskens's blog](https://thuijskens.github.io/2018/07/25/stability-selection/)

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.utils import check_random_state
from sklearn.feature_selection import SelectFromModel

from sklearn.metrics import accuracy_score

def stability_selection(X, 
                        y, 
                        model = LogisticRegression(l1_ratio = 1.0 , solver='liblinear'), 
                        param_name = "C",
                        param_values = np.logspace(-5,2,20), 
                        n_bootstrap_iterations = 100,
                        importance_getter = 'auto',
                        importance_thresold = 10**-5,
                        random_state = None):
    
    n_samples, n_variables = X.shape
    n_values = param_values.shape[0]

    rnd = check_random_state(random_state)
    
    selected_variables = np.zeros((n_variables,
                                 n_bootstrap_iterations))
    stability_scores = np.zeros((n_variables, n_values))

    for idx, param_value, in enumerate(param_values):
        
        # This is the sampling step, where bootstrap samples are generated
        # and the structure learner is fitted
        for iteration in range(n_bootstrap_iterations):
            
            bootstrap = rnd.choice(
                np.arange(n_samples),
                size=n_samples // 2,
                replace=False
            )

            X_train = X[bootstrap, :]
            y_train = y[bootstrap]

            # Assume scikit-learn implementation
            model.set_params(**{param_name: param_value}).fit(X_train, y_train)
            
            selected_variables[:, iteration] = SelectFromModel( 
                model , 
                prefit=True , 
                threshold = importance_thresold,
                importance_getter=importance_getter ).get_support()

        # This is the scoring step, where the final stability
        # scores are computed
        stability_scores[:, idx] = selected_variables.mean(axis=1)

    return stability_scores

In [ ]:
from sklearn.datasets import make_classification

data = make_classification(n_samples = 200 , 
                          n_features = 50 , n_informative = 5, n_redundant = 2 , 
                          return_X_y=False )

X,y = data['X'] , data['y']

data['feature_info']

In [ ]:
%%time
stability_scores = stability_selection( X , y ) 

The stability scores represent the fraction of bootstrap sample where the variable has a non-null importance for each values of the regularization parameter.

Each row corresponds to a feature, while the columns correspond to regularization parameter values.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.heatmap( stability_scores )
plt.xlabel("C")
plt.ylabel("variables")

The final support corresponds to the max of each row.

In [ ]:
stability_support = stability_scores.max(axis=1)
stability_support

In [ ]:
sns.stripplot( x = stability_support , y =  data['feature_info'] )

Given all this, it would make sense to select a sensible range for the regularization parameter.

Let's also do things a bit cleaner and add scaling.

---

In [ ]:
%%time
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
import pandas as pd

feature_names = pd.DataFrame(X).columns

logCs = []

coef_dict = {'name' : [],
             'val' : [],
             'log_C' : []}
accuracies = []

for C in np.logspace(-3,0,100):

    ppl = Pipeline([('scale' , StandardScaler()),
                ('model' , LogisticRegression(l1_ratio=1.0,
                                              C=C,
                                              solver="liblinear"))
               ])
    
    logCs.append(np.log10(C))
    accuracies.append( cross_val_score( ppl , X , y , scoring = 'accuracy').mean() )

    ppl.fit(X , y)
    
    coef_dict['name'] += list( feature_names )
    coef_dict['val'] += list( ppl['model'].coef_[0] )
    coef_dict['log_C'] += [np.log10(C)]* len(feature_names )

coef_df = pd.DataFrame(coef_dict)

bestC = logCs[ np.argmax( accuracies ) ]

fig,ax = plt.subplots(1,2,figsize = (15,7))

ax[0].plot(logCs , accuracies)
ax[0].set_xlabel("log10( C )")
ax[0].set_ylabel("cross-validated accuracy")
ax[0].axvline( bestC, color='r', ls = '--' )

sns.lineplot( x = 'log_C' , y='val' , hue = 'name' , data= coef_df , ax = ax[1] , legend=None)
ax[1].axvline( bestC , color='r', ls = '--' )
ax[1].set_ylabel( 'coefficient' )


fig.suptitle("logistic regression with an L1 regularization.")

min_logC_with_coef = coef_df.log_C[ coef_df.val != 0 ].min()

print(f"minimum log10(C) with a non-null coefficient: {min_logC_with_coef}")


In [ ]:
ppl = Pipeline([('scale' , StandardScaler()),
            ('model' , LogisticRegression(l1_ratio=1.0,solver="liblinear"))
           ])


stability_scores = stability_selection( X , y , 
                                       model = ppl , 
                                       param_name = "model__C",
                                       param_values= np.logspace(-1.7,1, 20),
                                       importance_getter= "named_steps.model.coef_") 

In [ ]:
sns.heatmap(stability_scores)

In [ ]:
stability_support = stability_scores.max(axis=1)
sns.stripplot( x = stability_support , y =  data['feature_info'] )

## applying this to the TGCA BRCA data

In [ ]:
from sklearn.cluster import AgglomerativeClustering

def drop_correlated_features( X , threshold = 0.9 ):
    """
    Args:
        - X (pd.DataFrame) : n,p feature matrix
        - threshold (float) : absolute correlation threshold group variables
    
    Returns:
        - pd.DataFrame : X with only the selected variables
        - dict : keys are the selected features , values are the list of features in the corresponding feature cluster 
    """
    
    corr_threshold = 0.9

    metric = 1 - X.corr().abs()

    HC = AgglomerativeClustering( n_clusters=None , metric='precomputed', linkage = 'single' , distance_threshold = (1-corr_threshold) )
    HC.fit(metric)

    variable_clusters = pd.Series( HC.labels_  , index = X.columns)

    cluster_to_features = variable_clusters.index.groupby(variable_clusters)

    ## keys are the selected feature in the cluster, values are the list of features in the cluster
    selected_features_to_features = { v[0]:list(v) for v in cluster_to_features.values() }

    return X.loc[:,selected_features_to_features.keys()] ,  selected_features_to_features 

In [ ]:
import pandas as pd
from sklearn.feature_selection import SelectPercentile
import numpy as np

## loading data
df_xpr = pd.read_csv("../data/TGCA_BRCA_expression_matrix.TPM.csv.gz" , index_col = 0)

df_clinical = pd.read_csv("../data/TGCA_BRCA_clinical_filtered.small.csv",index_col=0)
df_clinical = pd.get_dummies( df_clinical , drop_first=True)

## y is the poor_diagnosis
y = df_clinical.poor_prognosis

## ensuring the expression data is properly ordered
X_xpr = df_xpr.loc[ :, df_clinical.index].transpose() 

## selecting top 1% most variable genes
VT = SelectPercentile( score_func = lambda x,_ : np.var(x , axis = 0) ,
                       percentile = 1
                     )

X = pd.DataFrame( VT.fit_transform(X_xpr), columns=VT.get_feature_names_out() , index = X_xpr.index )
X , features_to_features_cluster = drop_correlated_features( X , threshold = 0.9 )


# adding age and sex to the set of features
X = pd.concat( [ df_clinical[['demographic.days_to_birth','demographic.sex_at_birth_male']] , X ] , axis=1 )

X.shape

In [ ]:
%%time
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
import pandas as pd

feature_names = pd.DataFrame(X).columns

logCs = []

coef_dict = {'name' : [],
             'val' : [],
             'log_C' : []}
accuracies = []

for C in np.logspace(-3,0,100):

    ppl = Pipeline([('scale' , StandardScaler()),
                ('model' , LogisticRegression(l1_ratio=1.0,
                                              C=C,
                                              solver="liblinear"))
               ])
    
    logCs.append(np.log10(C))
    accuracies.append( cross_val_score( ppl , X , y , scoring = 'accuracy').mean() )

    ppl.fit(X , y)
    
    coef_dict['name'] += list( feature_names )
    coef_dict['val'] += list( ppl['model'].coef_[0] )
    coef_dict['log_C'] += [np.log10(C)]* len(feature_names )

coef_df = pd.DataFrame(coef_dict)

bestC = logCs[ np.argmax( accuracies ) ]

fig,ax = plt.subplots(1,2,figsize = (15,7))

ax[0].plot(logCs , accuracies)
ax[0].set_xlabel("log10( C )")
ax[0].set_ylabel("cross-validated accuracy")
ax[0].axvline( bestC, color='r', ls = '--' )

sns.lineplot( x = 'log_C' , y='val' , hue = 'name' , data= coef_df , ax = ax[1] , legend=None)
ax[1].axvline( bestC , color='r', ls = '--' )
ax[1].set_ylabel( 'coefficient' )


fig.suptitle("logistic regression with an L1 regularization.")

min_logC_with_coef = coef_df.log_C[ coef_df.val != 0 ].min()

print(f"minimum log10(C) with a non-null coefficient: {min_logC_with_coef}")


In [ ]:
%%time
ppl = Pipeline([('scale' , StandardScaler()),
            ('model' , LogisticRegression(l1_ratio=1.0,solver="liblinear"))
           ])


stability_scores = stability_selection( np.array(X) , np.array(y) , 
                                       model = ppl , 
                                       param_name = "model__C",
                                       param_values= np.logspace(-1.7,-1.0, 20),
                                       importance_getter= "named_steps.model.coef_") 

In [ ]:
sns.heatmap(stability_scores)

In [ ]:
stability_support = pd.Series( stability_scores.max(axis = 1) , index = X.columns )
stability_support[ stability_support > 0.5 ]

## when in doubt: bootstrap, cross-validate, repeat

In general, it is a good idea test the robustness of any result by some form of repetition with variation.

This is the general concept underlying ensemble methods (ie, random forest), cross-validation, and as we have seen, boruta, knock-off aggregation methods or stability selection.

This can be extended to most cases.

Here is a simple example from [scikit-learn's documentation](https://scikit-learn.org/stable/auto_examples/inspection/plot_linear_model_coefficient_interpretation.html#checking-the-variability-of-the-coefficients)

In [ ]:
%%time
from sklearn.model_selection import RepeatedKFold, cross_validate

ppl = Pipeline([('scale' , StandardScaler()),
            ('model' , LogisticRegression(l1_ratio=1.0,solver="liblinear", C=10**bestC))
           ])


## training 5*5 models
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=0)

cv_model = cross_validate(
    ppl,
    X,
    y,
    cv=cv,
    return_estimator=True,
    n_jobs=2,
)

In [ ]:
coefs_cv = pd.DataFrame([ train_ppl['model'].coef_[0] for train_ppl in cv_model['estimator'] ] ,
            columns = X.columns )

coefs_cv.head()

In [ ]:
## keeping only features which have at least 1 non-zero parameter
sometimes_not_0 = (coefs_cv.abs().max() > 0)
coefs_cv = coefs_cv.loc[:,sometimes_not_0]

## plotting 

plt.figure(figsize=(9, 7))
sns.stripplot(data=coefs_cv, orient="h", palette="dark:k", alpha=0.5)
sns.boxplot(data=coefs_cv, orient="h", color="cyan", saturation=0.5, whis=10)
plt.axvline(x=0, color=".5")
plt.xlabel("Coefficient importance")
plt.title("Coefficient importance and its variability")
